# 📄 Synthetic Documents for AI Agents

In this lesson, we are going to generate realistic synthetic documents, from a single invoice to a diverse batch, that can safely emulate real business workflows for AI agent development and testing.

So far in this chapter we've generated structured tabular data. But modern AI systems, agents, extractors, classifiers, interact primarily with **unstructured documents**: invoices, reports, contracts, receipts. Generating realistic synthetic versions of these makes it possible to build and test AI pipelines without exposing sensitive business data.

1. Cover five design principles for synthetic document generation
2. Generate a single synthetic invoice using `DocumentGenerator`
3. Generate a diverse batch using variability controls
4. Generate from an existing document template image

## 1. Five Design Principles

Before generating a single document, it is worth thinking through what makes synthetic documents *useful* for AI systems. A document that looks plausible to a human is not necessarily one that trains or tests an AI agent well.

**1. Define the document's purpose.**  
What task should the AI perform with this document? Extract totals? Identify line items? Classify the document type? The purpose determines what fields must be present and realistic.

**2. Structural realism.**  
A synthetic supermarket invoice should follow the same logic as a real one: line items before subtotals, tax applied to the right base, dates that make sense in sequence. An AI trained on structurally broken documents learns broken patterns.

**3. Linguistic realism.**  
Terminology, tone, and communication style should reflect the domain. A legal contract and a retail receipt use different language — and so should their synthetic counterparts.

**4. Document diversity.**  
Real-world documents vary in vendor, format, region, and style. A diverse synthetic dataset prevents AI systems from overfitting to a single layout or phrasing pattern.

**5. Format alignment.**  
Choose the output format — PDF, Word (.docx), or HTML — to match the format your target system will actually process. A document ingestion pipeline that handles PDFs should be tested on PDFs, not HTML.

## 2. Generate a Single Document

The `DocumentGenerator` class is the entry point for all document synthesis. Each of the parameters maps directly to one of the design principles above — purpose, tone, structure, language, and format are all explicit inputs, not left to chance.

In [1]:
from ydata.synthesizers.text.model.document import DocumentGenerator, DocumentFormat

# Initialise the generator — choose the output format here
# Supported: DocumentFormat.PDF | DocumentFormat.DOCX | DocumentFormat.HTML
generator = DocumentGenerator(document_format=DocumentFormat.PDF,
                              backend='anthropic',
                              subscription_key='add-your-subscription-key',
                              model='claude-sonnet-4-5')
print('DocumentGenerator ready ✓')

INFO: 2026-08-11 14:21:54,743 Using device of type: cpu
DocumentGenerator ready ✓


In [2]:
# Generate a single supermarket invoice
# Each parameter steers a different dimension of the output
generator.generate(
    n_docs=1,
    document_type='Invoice',
    audience='Retail customer',
    tone='neutral',           # formal | casual | persuasive | empathetic |
                               # inspirational | enthusiastic | humorous | neutral
    purpose='Detailed supermarket invoice with grocery and household item purchases.',
    region='North America',
    language='English',
    length='Long',            # Long allows for realistic number of line items
    topics='Groceries, Household goods, Unit price, Quantity, Subtotals, Tax, Total due, Payment method',
    style_guide='Clean and readable receipt-style format typical of supermarket invoices',
    output_dir='output/documents',
)

print('Document generated → output/documents/')

INFO: 2026-08-11 14:23:59,215 HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO: 2026-08-11 14:24:27,697 HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO: 2026-08-11 14:26:45,430 HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

Open the file in `output/documents/` and review it. Notice how the parameters translate into content decisions:

- `purpose` determines what fields appear (line items, tax, totals)
- `topics` controls what categories of items are included
- `style_guide` shapes the visual layout and typography choices
- `region` and `language` influence currency symbols, date formats, and phrasing

> **⏸ Active Learning** — Pause here. Think of a document type you'd find useful in your own domain — a contract, a medical report, a shipping manifest. Change the parameters above and generate your own synthetic document before continuing.

## 3. Document Variability

A single document is useful for a quick test. But training or evaluating an AI agent requires *diversity* — documents that vary in vendor, format, tone, and content, just as real documents do.

### 3a. Simple batch — same parameters, multiple documents

The simplest form of variability is just generating more: the model introduces natural variation in content while keeping the structural parameters fixed.

In [ ]:
# Generate 5 invoices — same configuration, natural content variation
generator.generate(
    n_docs=5,
    document_type='Invoice',
    audience='Retail customer',
    tone='neutral',
    purpose='Detailed supermarket invoice with grocery and household item purchases.',
    region='North America',
    language='English',
    length='Long',
    topics='Groceries, Household goods, Unit price, Quantity, Subtotals, Tax, Total due, Payment method',
    style_guide='Clean and readable receipt-style format typical of supermarket invoices',
    output_dir='output/documents/batch',
)

print('Batch generated → output/documents/batch/')

### 3b. Controlled variability — `DatasetConfig`

For more precise control over diversity, `DatasetConfig` lets you define **profiles** and **weighted distributions** across parameters. This is how you build a dataset where 50% of invoices come from supermarkets, 30% from consulting firms, and 20% from utilities — all in a single call.

In [ ]:
from ydata.synthesizers.text.model.utils.dataset_config import DatasetConfig

config = DatasetConfig(
    document_type='Invoice',

    # Base: parameters shared across all generated documents
    base={
        'language': 'English',
        'region':   'North America',
        'length':   'Long',
    },

    # Variations: control how documents differ from each other
    variations={
        # Weighted profiles — 50% supermarket, 30% consulting, 20% utility
        'profile': [
            {
                'weight':   0.5,
                'audience': 'Retail customer',
                'purpose':  'Supermarket invoice with grocery and household item purchases.',
                'topics':   'Groceries, Household goods, Unit price, Quantity, Subtotals, Tax, Total due',
            },
            {
                'weight':   0.3,
                'audience': 'Corporate client',
                'purpose':  'Invoice for professional consulting services rendered.',
                'topics':   'Consulting services, Hourly rates, Tax breakdown, Payment terms',
            },
            {
                'weight':   0.2,
                'audience': 'Residential customer',
                'purpose':  'Monthly utility bill for electricity and gas consumption.',
                'topics':   'Energy usage, Meter readings, Tariff, Taxes, Total due',
            },
        ],
        # Tone drawn from a weighted distribution
        'tone': {'formal': 0.6, 'neutral': 0.4},
    },

    # Constraints: structural rules the model must respect
    constraints={
        'must_include': ['total', 'tax'],
        'min_items': 5,
    },
)

metadata = generator.generate_dataset(
    config=config,
    n_docs=10,
    output_dir='output/documents/varied',
    seed=42,
)

print('Varied dataset generated → output/documents/varied/')
print(metadata)

---
## 4. Template-Based Generation

Sometimes you need synthetic documents that match a **specific layout** — not just realistic content, but the same visual structure as an existing form or template. This is common when testing systems that rely on layout cues: OCR pipelines, form extractors, or document classifiers trained on a particular format.

`generate_from_template()` takes a reference image of a real document (a screenshot or scan), extracts its structure, and uses it as the scaffold for the synthetic output.

The `information` parameter accepts three forms:
- A **dict** — structured data to populate the template (most control)
- A **list of dicts** — batch generation, one document per entry
- A **string** — a natural language description, when you don't have structured data

In [ ]:
# Template-based generation from a reference image
# Replace 'assets/invoice_template.png' with the path to your template image
# Supported template formats: PNG, JPG

generator.generate_from_template(
    document_type='Invoice',
    information={
        'invoice_number': 'INV-2025-00142',
        'issue_date':     '2025-03-01',
        'due_date':       '2025-03-31',
        'payment_terms':  'Net-30',
        'seller': {
            'name':    'Fresh Mart Inc.',
            'address': '100 Market Street, Austin, TX 78701',
            'tax_id':  'US-84-9876543',
            'email':   'billing@freshmart.com',
        },
        'bill_to': {
            'name':    'Jane Smith',
            'address': '42 Maple Avenue, Austin, TX 78702',
            'email':   'jane.smith@email.com',
        },
        'line_items': [
            {'description': 'Organic whole milk (1 gal)',  'qty': 2,  'unit_price': 5.49,  'total': 10.98},
            {'description': 'Sourdough bread loaf',       'qty': 1,  'unit_price': 4.29,  'total': 4.29},
            {'description': 'Free-range eggs (12 ct)',    'qty': 1,  'unit_price': 6.99,  'total': 6.99},
            {'description': 'Baby spinach (5 oz)',        'qty': 2,  'unit_price': 3.49,  'total': 6.98},
            {'description': 'Dish soap (32 oz)',          'qty': 1,  'unit_price': 3.79,  'total': 3.79},
        ],
        'subtotal':   32.03,
        'tax_rate':   0.08,
        'tax_amount': 2.56,
        'total_due':  34.59,
        'currency':   'USD',
        'notes':      'Thank you for shopping at Fresh Mart!',
    },
    template_img_path='assets/invoice_template.png',  # ← your reference image
    output_dir='output/documents/from_template',
)

print('Template-based document generated → output/documents/from_template/')

Template-based generation is particularly useful when testing systems that rely on layout, formatting, and visual cues. Unlike free-form generation, it preserves the structural skeleton of the original — column positions, section order, header styles — while filling in entirely synthetic content.

You can also pass a logo to brand the output:

```python
generator.generate_from_template(
    ...,
    logo_path='assets/freshmart_logo.png',  # PNG, JPG, GIF, SVG, or WEBP
)
```

## ✅ Summary

In this notebook you:

1. **Covered five design principles** for synthetic document generation — purpose, structural realism, linguistic realism, diversity, and format alignment
2. **Generated a single synthetic invoice** using `DocumentGenerator.generate()`, with explicit control over audience, tone, purpose, region, language, and style
3. **Generated a diverse batch** using `DatasetConfig` — weighted vendor profiles, tone distributions, and structural constraints in a single call
4. **Generated from a template image** using `generate_from_template()` — preserving the visual structure of a reference document while filling in synthetic content

The key insight: synthetic documents are not just filled-in templates. The parameters you pass — purpose, tone, topics, constraints — encode domain knowledge that shapes what the AI generates. The more precisely you describe the document, the more useful the output is for training and testing AI systems.

> **Active Learning:** Try changing `document_format` from `DocumentFormat.PDF` to `DocumentFormat.DOCX` and regenerate. Then try a different `document_type` — `'Contract'`, `'Report'`, or `'Purchase Order'` — and observe how the same parameters produce structurally different outputs.